# UTMIST ML Seed — Exploration Notebook

This notebook walks through the core components: loading configs, building models, preparing data, and running a short training loop.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import os

CONFIGS_DIR = os.path.abspath("../configs")

## 1. Load config via Hydra Compose API

In [ ]:
with initialize_config_dir(config_dir=CONFIGS_DIR, version_base=None):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

## 2. Build model and inspect architecture

In [ ]:
from src.models import build_model

model = build_model(cfg.model)
print(f"Model: {cfg.model.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print()

# Quick forward pass
x = torch.randn(1, 3, 32, 32)
out = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

## 3. Build dataset and visualize samples

In [ ]:
from src.data import build_dataset, build_dataloaders

train_ds, val_ds = build_dataset(cfg.dataset)
train_loader, val_loader = build_dataloaders(train_ds, val_ds, batch_size=16)

print(f"Train samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")

# Visualize a batch
cifar_classes = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * [0.2470, 0.2435, 0.2616] + [0.4914, 0.4822, 0.4465]  # unnormalize
    img = img.clip(0, 1)
    ax.imshow(img)
    ax.set_title(cifar_classes[labels[i]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Train for 2 epochs

In [ ]:
import torch.nn as nn
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(cfg.model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.lr)
criterion = nn.CrossEntropyLoss()

train_losses = []

for epoch in range(1, 3):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch}"):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)
        correct += outputs.argmax(1).eq(targets).sum().item()
        total += inputs.size(0)

    avg_loss = epoch_loss / total
    acc = correct / total
    train_losses.append(avg_loss)
    print(f"Epoch {epoch}: loss={avg_loss:.4f}, acc={acc:.4f}")

## 5. Plot training loss

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()